In [ ]:
!python3 -m pip install requests

In [2]:
# ---------------------------------------
# IMPORTS
# ---------------------------------------
import time
import csv
from typing import Dict, Any, List
from datetime import datetime, timedelta, timezone
import requests

In [34]:
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv(Path("..") / ".env")
ACCESS_TOKEN = os.getenv('USER_TOKEN')

In [4]:
CREATION_DATE = '2025-06-25'

In [11]:
import requests
import csv
from datetime import datetime, timezone, timedelta

# Limpia token por si acaso (saltos de línea, espacios)
ACCESS_TOKEN = "".join(str(ACCESS_TOKEN).split())

IG_USER_ID = "17841467002220104"
CREATION_DATE = "2024-06-25"  # <-- pon tu fecha real (YYYY-MM-DD)

METRICS = [
    "website_clicks",
    "profile_views",
    "accounts_engaged",
    "total_interactions",
    "likes",
    "comments",
    "shares",
    "saves",
    "replies",
    "follows_and_unfollows"
]
VERSION = "v19.0"

def totals_since_creation_30d(ig_user_id, access_token, metrics, creation_date, version="v19.0"):
    url = f"https://graph.facebook.com/{version}/{ig_user_id}/insights"

    totals = {m: 0 for m in metrics}

    cur = datetime.strptime(creation_date, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    end = datetime.now(timezone.utc)

    while cur < end:
        nxt = min(cur + timedelta(days=30), end)

        params = {
            "metric": ",".join(metrics),
            "period": "day",
            "metric_type": "total_value",
            "since": int(cur.timestamp()),
            "until": int(nxt.timestamp()),
            "access_token": access_token
        }

        r = requests.get(url, params=params, timeout=60)
        data = r.json()

        if r.status_code != 200:
            raise RuntimeError(f"Fallo ventana {cur.date()} → {nxt.date()}: {data}")

        for obj in data.get("data", []):
            name = obj.get("name")
            val = (obj.get("total_value") or {}).get("value", 0)
            if name in totals and isinstance(val, (int, float)):
                totals[name] += val

        cur = nxt

    return totals

# 1) Calcula totales
totals = totals_since_creation_30d(
    ig_user_id=IG_USER_ID,
    access_token= ACCESS_TOKEN,
    metrics=METRICS,
    creation_date=CREATION_DATE,
    version=VERSION
)

# 2) Guarda a CSV
with open("ig_totals_since_creation.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["metric", "value"])
    for k, v in totals.items():
        w.writerow([k, v])

print("✅ Guardado ig_totals_since_creation.csv")
totals

✅ Guardado ig_totals_since_creation.csv


{'website_clicks': 543,
 'profile_views': 2213,
 'accounts_engaged': 214,
 'total_interactions': 258,
 'likes': 137,
 'comments': 12,
 'shares': 30,
 'saves': 33,
 'replies': 11,
 'follows_and_unfollows': 0}

In [12]:
import requests
import csv
from datetime import datetime, timezone, timedelta

ACCESS_TOKEN = "".join(str(ACCESS_TOKEN).split())
IG_USER_ID = "17841467002220104"
CREATION_DATE = "2024-06-25"
VERSION = "v19.0"

METRICS = [
    "website_clicks",
    "profile_views",
    "accounts_engaged",
    "total_interactions",
    "likes",
    "comments",
    "shares",
    "saves",
    "replies",
    "follows_and_unfollows"
]

def totals_by_window_30d_long(ig_user_id, access_token, metrics, creation_date, version="v19.0"):
    url = f"https://graph.facebook.com/{version}/{ig_user_id}/insights"

    rows = []
    cur = datetime.strptime(creation_date, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    end = datetime.now(timezone.utc)

    while cur < end:
        nxt = min(cur + timedelta(days=30), end)

        params = {
            "metric": ",".join(metrics),
            "period": "day",
            "metric_type": "total_value",
            "since": int(cur.timestamp()),
            "until": int(nxt.timestamp()),
            "access_token": access_token
        }

        r = requests.get(url, params=params, timeout=60)
        data = r.json()

        if r.status_code != 200:
            raise RuntimeError(f"Fallo ventana {cur.date()} → {nxt.date()}: {data}")

        window_start = cur.strftime("%Y-%m-%d")
        window_end = nxt.strftime("%Y-%m-%d")

        # Guardamos solo las métricas que devuelve la API en esa ventana
        for obj in data.get("data", []):
            rows.append({
                "window_start": window_start,
                "window_end": window_end,
                "metric": obj.get("name"),
                "value": (obj.get("total_value") or {}).get("value")
            })

        cur = nxt

    return rows

# Ejecuta
rows_long = totals_by_window_30d_long(IG_USER_ID, ACCESS_TOKEN, METRICS, CREATION_DATE, VERSION)

# Guarda CSV (formato largo)
with open("ig_totals_by_window_long.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["window_start", "window_end", "metric", "value"])
    w.writeheader()
    w.writerows(rows_long)

print("✅ Guardado ig_totals_by_window_long.csv")

✅ Guardado ig_totals_by_window_long.csv


In [16]:
import requests

def fetch_follower_demographics(breakdown: str):
    url = f"https://graph.facebook.com/{VERSION}/{IG_USER_ID}/insights"
    params = {
        "metric": "follower_demographics",
        "period": "lifetime",
        "metric_type": "total_value",
        "breakdown": breakdown,
        "access_token": ACCESS_TOKEN
    }

    r = requests.get(url, params=params, timeout=60)
    data = r.json()

    if r.status_code != 200:
        raise RuntimeError(f"Fallo follower_demographics({breakdown}): {data}")

    # NUEVA ESTRUCTURA: total_value -> breakdowns -> results
    try:
        breakdowns = data["data"][0]["total_value"]["breakdowns"]
        # buscamos el breakdown que coincide (por si vinieran varios)
        for b in breakdowns:
            if breakdown in b.get("dimension_keys", []):
                out = {}
                for res in b.get("results", []):
                    key = res.get("dimension_values", ["unknown"])[0]
                    out[key] = res.get("value", 0)
                return out
        return {}
    except (KeyError, IndexError, TypeError):
        return {}

In [17]:
import csv

for b in ["age", "gender", "country", "city"]:
    demo = fetch_follower_demographics(b)

    with open(f"ig_follower_demographics_{b}.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["breakdown", "category", "value"])
        for cat, val in demo.items():
            w.writerow([b, cat, val])

    print(f"✅ Guardado ig_follower_demographics_{b}.csv")

✅ Guardado ig_follower_demographics_age.csv
✅ Guardado ig_follower_demographics_gender.csv
✅ Guardado ig_follower_demographics_country.csv
✅ Guardado ig_follower_demographics_city.csv
